#### Schema Evolution 
   - **Ability to handle changes in Schema without having to comletely re-write your table**

1. Adding new column (Manual/Automatic)
2. Widening Data Types (Supported delta>=3.2): Sometimes we need to expand a column's data type to accomodate larger values. Delta Lake allows "widening" type conversions that won't lose data, such as:

   - INT to BIGINT
   - float to DOUBLE
   - VARCHAR(10) to VARCHAR(20)

3. Nested Structure Evolution (Manual / Automatic) : Delta Lake supports evolution of complex data types like structs and arrays. We can:
   - Add new fields to structs.
   - Modify nested field types.
   - Add new elements to arrays.

4. Column Positioning Changes (Manual/Automatic): We can re-organize our columns.

**Note :** 
   - **INSERT** works by matching columns by position.
   - **MERGE** works by matching columns by name.

**Table Set Up**

In [0]:
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_se
(
  customer_id INT NOT NULL,
  invoice_no STRING,
  price FLOAT,
  invoice_date DATE
);

INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price , invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 1 AND 5;

In [0]:
SELECT * from deltacatalog.deltadb.invoices_se;

#### Scenario 1 : Adding New Columns

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ADD COLUMNS(
  Quantity INT
);

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price , invoice_date, quantity
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 6 AND 10;

In [0]:
Select * from deltacatalog.deltadb.invoices_se;

In [0]:
SELECT *
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet` limit 2;

Here we adjusted and added a new column by doing ALTER Command. What if we want to do it automatically?

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled = true;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price , invoice_date, quantity, payment_method
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 11 AND 15;

In [0]:
Select * from deltacatalog.deltadb.invoices_se;

#### Scenario 2 : Type Widening

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');

In [0]:
DESCRIBE TABLE deltacatalog.deltadb.invoices_se;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
VALUES (123456789012345, 'I106485', 30.29999999908008238, '2022-12-01', 2, 'Debit-Card')

In [0]:
Select * from deltacatalog.deltadb.invoices_se;

In [0]:
DESCRIBE TABLE deltacatalog.deltadb.invoices_se;

**Automatic Type Widening Works fine.**

#### Scenario 3 : Nested Structure Evolution

In [0]:
ALTER Table deltacatalog.deltadb.invoices_se
ADD COLUMNS purchase_details STRUCT<
  mall_pin_code INT,
  store_code INT
>;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
VALUES (16, 'I106485', 30.29999999908008238, '2022-12-01', 2, 'Debit-Card', STRUCT(12345,879));

In [0]:
select * from deltacatalog.deltadb.invoices_se;

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ALTER COLUMN purchase_details.mall_pin_code TYPE BIGINT;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
VALUES (17, 'I106485', 30.29999999908008238, '2022-12-01', 2, 'Debit-Card', STRUCT(123456789012345,7890));

In [0]:
Select * from deltacatalog.deltadb.invoices_se;

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ADD COLUMN purchase_details.store_loc STRING; 

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
VALUES (17, 'I106485', 30.29999999908008238, '2022-12-01', 2, 'Debit-Card', STRUCT(7612,7890,'Lucknow'));

In [0]:
select * from deltacatalog.deltadb.invoices_se;

With Schema Evolution

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
VALUES (21, 'I106485', 30.29999999908008238, '2022-12-01', 2, 'Debit-Card', 
  NAMED_STRUCT(
    'mall_pin_code', 7612,
    'store_code', 765,
    'store_loc','ground floor',
    'staff_id','ST12736'
  )
);

In [0]:
Select * from deltacatalog.deltadb.invoices_se;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
VALUES (22, 'I106485', 30.29999999908008238, '2022-12-01', 2, 'Debit-Card', 
  NAMED_STRUCT(
    'mallPinCode', 7612,
    'storeCode', 765,
    'storeLoc','ground floor',
    'staffId','ST12736',
    'shopName','Sephora'
  )
);

In [0]:
Select * from deltacatalog.deltadb.invoices_se;

#### Scenario 4: Column Position Change

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled = false;

In [0]:
-- ALTER TABLE deltacatalog.deltadb.invoices_se ADD COLUMNS (age INT FIRST);
ALTER TABLE deltacatalog.deltadb.invoices_se ADD COLUMNS (age INT AFTER price);

In [0]:
Select * from deltacatalog.deltadb.invoices_se;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price , age, invoice_date, quantity, payment_method,NULL
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 50 AND 55;

In [0]:
select * from deltacatalog.deltadb.invoices_se;

With Schema Evolution Enabled

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled = true;

- Insert fails
- Error : Cannot resolve "category" due to data type mismatch: cannot cast "STRING" to "STRUCT<mall_pin_code: BIGINT, store_code: INT, store_loc: STRING, staff_id: STRING, shopName: STRING>"

In [0]:
%skip
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price , age, invoice_date, quantity, payment_method, category, NULL as purchase_details
FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 56 AND 60;

In [0]:
MERGE INTO deltacatalog.deltadb.invoices_se tgt
USING 
(
  SELECT customer_id, invoice_no, price , age, invoice_date, quantity, payment_method, category, NULL as purchase_details
  FROM PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE customer_id BETWEEN 56 AND 60
) src
ON tgt.customer_id = src.customer_id
WHEN NOT MATCHED THEN
INSERT *


_ Merge does matching by name, so it inserted it but positioning didnt work.

In [0]:
select * from deltacatalog.deltadb.invoices_se;

In [0]:
%python

from pyspark.sql.functions import col

df = (
    spark.read.parquet("abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet")
    .filter(col("customer_id").between(1, 10))
    .select("customer_id","price","invoice_date")
)

df.write.saveAsTable("deltacatalog.deltadb.invoices_se_spark_df")

In [0]:
%python
df = (spark.read.parquet("abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet")
      .filter(col("customer_id").between(11,25))
      .select("customer_id","price","invoice_date","quantity","payment_method")
      )

df.write.mode("append").option("mergeSchema","true").saveAsTable("deltacatalog.deltadb.invoices_se_spark_df")

In [0]:
select * from deltacatalog.deltadb.invoices_se_spark_df